# Train a Policy with an Ensemble Time2Success Reward

Changes from v2:
- **Ensemble reward** — 5 models, reward penalized by their disagreement
  (`mean + w·std`), so uncertain regions look farther from success. Based on
  the diagnostic showing 3-5× higher std on the bad seeds.
- **Fewer checkpoints** — `ckpt_freq=500_000` (6 files, not 1500).
- **Hang diagnostics** — timestamped prints around save/close, and
  `progress_bar=False`, to localize the end-of-training stall.

Pure model-derived reward only: no dense reward, no ground-truth bonus.

## 1. Setup

In [9]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import metaworld
import imageio
import collections
import json, os, time
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback

TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode)

_probe = make_env(seed=0)
DT = getattr(_probe.unwrapped, "dt", _probe.unwrapped.model.opt.timestep)
OBS_DIM = _probe.observation_space.shape[0]
_probe.close()
print(f"DT={DT}, OBS_DIM={OBS_DIM}")

# dropout=0.2 MUST be here — the ensemble checkpoints were trained with it,
# and a mismatched architecture fails to load with a state_dict key error.
class Time2SuccessModel(nn.Module):
    def __init__(self, obs_dim, hidden=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = "cuda" if torch.cuda.is_available() else "cpu"

DT=0.0125, OBS_DIM=39


d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")


## 2. Load the frozen ensemble

In [10]:
ENSEMBLE_DIR = "./checkpoints/time2success_ensemble"
N_ENSEMBLE = 5

norm = np.load(f"{ENSEMBLE_DIR}/normalization.npz")
SHAPING_X_MEAN, SHAPING_X_STD = norm["X_mean"], norm["X_std"]
SHAPING_Y_MEAN, SHAPING_Y_STD = norm["y_mean"].item(), norm["y_std"].item()

shaping_ensemble = []
for i in range(N_ENSEMBLE):
    m = Time2SuccessModel(obs_dim=OBS_DIM, dropout=0.2).to(device)
    m.load_state_dict(torch.load(f"{ENSEMBLE_DIR}/model_{i}.pt"))
    m.eval()   # dropout OFF — disagreement must come from different fits,
               # not from random dropout masks
    shaping_ensemble.append(m)

print(f"Loaded {len(shaping_ensemble)} members from {ENSEMBLE_DIR}")

# Quick scale check — std here informs uncertainty_weight below
_x = torch.tensor((make_env(seed=0).reset()[0] - SHAPING_X_MEAN) / SHAPING_X_STD,
                   dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    _p = np.array([m(_x).item() for m in shaping_ensemble]) * SHAPING_Y_STD + SHAPING_Y_MEAN
print(f"At a reset state: mean={_p.mean():.1f}, std={_p.std():.1f} steps")

Loaded 5 members from ./checkpoints/time2success_ensemble
At a reset state: mean=57.9, std=3.9 steps


## 3. Ensemble reward wrapper

`_potential` returns `-(mean + w·std)`. Higher disagreement makes a state
look farther from success, so the policy is pulled toward regions the
ensemble collectively understands.

In [16]:
GAMMA = 0.99

class EnsembleTime2SuccessRewardWrapper(gym.Wrapper):
    def __init__(self, env, t2s_models, device, x_mean, x_std, y_mean, y_std,
                 gamma=GAMMA, shaping_scale=1.0, stall_window=10_000,
                 max_pred_steps=500, uncertainty_weight=0.5):
        super().__init__(env)
        self.t2s_models = t2s_models          # list of 5, not one model
        self.device = device
        self.x_mean, self.x_std = x_mean, x_std
        self.y_mean, self.y_std = y_mean, y_std
        self.gamma = gamma
        self.shaping_scale = shaping_scale
        self.stall_window = stall_window
        self.max_pred_steps = max_pred_steps
        self.uncertainty_weight = uncertainty_weight
        self._last_phi = 0.0
        self._recent_predictions = collections.deque(maxlen=self.stall_window)

    @torch.no_grad()
    def _potential(self, obs):
        x_norm = (obs - self.x_mean) / self.x_std
        x = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(self.device)
        preds = np.array([m(x).item() for m in self.t2s_models])
        preds = preds * self.y_std + self.y_mean
        penalized = preds.mean() + self.uncertainty_weight * preds.std()
        return -float(np.clip(penalized, 0, self.max_pred_steps))

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_phi = self._potential(obs)
        self._recent_predictions.clear()
        self._recent_predictions.append(-self._last_phi)
        return obs, info

    def step(self, action):
        obs, _base_reward, terminated, truncated, info = self.env.step(action)  # env reward discarded
        phi_next = self._potential(obs)
        predicted_now = -phi_next

        if (len(self._recent_predictions) == self.stall_window
                and predicted_now >= max(self._recent_predictions)):
            truncated = True

        self._recent_predictions.append(predicted_now)
        pure_t2s_reward = self.gamma * phi_next - self._last_phi
        self._last_phi = phi_next
        return obs, self.shaping_scale * pure_t2s_reward, terminated, truncated, info

def make_pure_t2s_env(seed=0, uncertainty_weight=1.5):
    return EnsembleTime2SuccessRewardWrapper(
        make_env(seed=seed), shaping_ensemble, device,
        SHAPING_X_MEAN, SHAPING_X_STD, SHAPING_Y_MEAN, SHAPING_Y_STD,
        uncertainty_weight=uncertainty_weight,
    )

**Reward magnitude check** — worth running before committing hours.
Compare typical per-step reward against the std penalty scale.

In [17]:
_e = make_pure_t2s_env(seed=0)
_obs, _ = _e.reset()
_rewards = []
for _t in range(30):
    _obs, _r, _term, _trunc, _ = _e.step(_e.action_space.sample())
    _rewards.append(_r)
    if _term or _trunc:
        break
_e.close()
_rewards = np.array(_rewards)
print(f"Random-action rewards over {len(_rewards)} steps: "
      f"mean={_rewards.mean():.3f}, min={_rewards.min():.3f}, max={_rewards.max():.3f}")
print("If these are ~0 or enormous, adjust shaping_scale before the full run.")

Random-action rewards over 30 steps: mean=0.246, min=-1.267, max=1.754
If these are ~0 or enormous, adjust shaping_scale before the full run.


## 4. Training setup

In [18]:
N_ENVS = 6
UNCERTAINTY_WEIGHT = 1.5   # std ~1-2 on good states, ~10-12 on bad ones

pure_train_env = DummyVecEnv(
    [lambda i=i: make_pure_t2s_env(seed=i, uncertainty_weight=UNCERTAINTY_WEIGHT)
     for i in range(N_ENVS)])
pure_train_env = VecMonitor(pure_train_env)
pure_eval_env = make_env(seed=1000)   # raw env — real success flag, never shaped

pure_model = SAC(
    policy="MlpPolicy",
    env=pure_train_env,
    learning_rate=3e-4,
    buffer_size=1_000_000,
    batch_size=256,
    tau=0.005,
    gamma=GAMMA,
    ent_coef="auto",
    policy_kwargs=dict(net_arch=[400, 400]),
    tensorboard_log="./tb_logs/peg_insert_side_ensemble_t2s",
    verbose=1,
    seed=0,
)

Using cpu device


In [19]:
class SuccessCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=25_000, n_eval_episodes=10,
                 ckpt_dir="./checkpoints/peg_insert_side_ensemble_t2s",
                 ckpt_freq=500_000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.ckpt_dir = ckpt_dir
        self.ckpt_freq = ckpt_freq
        os.makedirs(ckpt_dir, exist_ok=True)
        self.history = []

    def _run_eval_episode(self):
        obs, _ = self.eval_env.reset()
        success_step = None
        for t in range(500):
            action, _ = self.model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = self.eval_env.step(action)
            if info.get(SUCCESS_KEY, 0) and success_step is None:
                success_step = t
            if terminated or truncated:
                break
        return success_step

    def _on_step(self) -> bool:
        if self.num_timesteps % self.ckpt_freq < self.training_env.num_envs:
            self.model.save(os.path.join(self.ckpt_dir, f"ens_t2s_{self.num_timesteps}.zip"))
        if self.num_timesteps % self.eval_freq < self.training_env.num_envs:
            steps = [self._run_eval_episode() for _ in range(self.n_eval_episodes)]
            success_rate = sum(s is not None for s in steps) / self.n_eval_episodes
            times = [s for s in steps if s is not None]
            mean_t2s = float(np.mean(times)) if times else None
            self.logger.record("eval/success_rate", success_rate)
            if mean_t2s is not None:
                self.logger.record("eval/mean_time_to_success", mean_t2s)
            self.history.append(dict(step=self.num_timesteps, success_rate=success_rate,
                                       mean_time_to_success=mean_t2s, timestamp=time.time()))
            with open(os.path.join(self.ckpt_dir, "eval_history.json"), "w") as f:
                json.dump(self.history, f, indent=2)
            if self.verbose:
                print(f"[{time.strftime('%H:%M:%S')}] eval @ {self.num_timesteps}: "
                      f"success_rate={success_rate:.2f} mean_t2s={mean_t2s}")
        return True

# Smoke-test frequencies; the full run overrides these in section 6
pure_callback = SuccessCallback(eval_env=pure_eval_env, ckpt_freq=2000, eval_freq=2000)

## 5. Smoke test

Five forward passes per env step instead of one — note the wall-clock time
here and extrapolate before committing to 3M steps.

In [20]:
_t0 = time.time()
pure_model.learn(total_timesteps=5_000, callback=pure_callback, tb_log_name="ens_t2s_smoke")
_elapsed = time.time() - _t0
print(f"\nSmoke test: {_elapsed:.1f}s for 5,000 steps")
print(f"Extrapolated to 3M steps: {_elapsed * 600 / 3600:.1f} hours")
print("If that is unacceptable, reduce N_ENSEMBLE or N_ENVS before the full run.")

Logging to ./tb_logs/peg_insert_side_ensemble_t2s\ens_t2s_smoke_3
[20:28:33] eval @ 2004: success_rate=0.00 mean_t2s=None
---------------------------------
| eval/              |          |
|    success_rate    | 0        |
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 299      |
| time/              |          |
|    episodes        | 4        |
|    fps             | 179      |
|    time_elapsed    | 16       |
|    total_timesteps | 3000     |
| train/             |          |
|    actor_loss      | -10.3    |
|    critic_loss     | 0.477    |
|    ent_coef        | 0.865    |
|    ent_coef_loss   | -0.977   |
|    learning_rate   | 0.0003   |
|    n_updates       | 483      |
---------------------------------
[20:28:45] eval @ 4002: success_rate=0.00 mean_t2s=None

Smoke test: 29.4s for 5,000 steps
Extrapolated to 3M steps: 4.9 hours
If that is unacceptable, reduce N_ENSEMBLE or N_ENVS before the full run.


## 6. Full run

In [21]:
TOTAL_TIMESTEPS = 3_000_000
CKPT_DIR = "./checkpoints/peg_insert_side_ensemble_t2s_2"

# Rebuild the callback with production frequencies:
#   6 checkpoints instead of 1500, 120 eval points instead of 1500.
pure_callback = SuccessCallback(eval_env=pure_eval_env,
                                 ckpt_freq=500_000, eval_freq=25_000)

print(f"[{time.strftime('%H:%M:%S')}] starting learn()")
pure_model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=pure_callback,
    tb_log_name="ens_t2s_run2",
    progress_bar=False,   # tqdm/rich can hang the cell at completion; TensorBoard covers monitoring
    reset_num_timesteps=False,
)
print(f"[{time.strftime('%H:%M:%S')}] learn() RETURNED")

pure_model.save(os.path.join(CKPT_DIR, "sac_peg_insert_ensemble_t2s_final"))
print(f"[{time.strftime('%H:%M:%S')}] MODEL SAVED — safe to interrupt from here on")

# Close individually so a hang is attributable to a specific call
try:
    pure_eval_env.close()
    print(f"[{time.strftime('%H:%M:%S')}] eval env closed")
except Exception as e:
    print("eval env close failed:", e)

try:
    pure_train_env.close()
    print(f"[{time.strftime('%H:%M:%S')}] train env closed")
except Exception as e:
    print("train env close failed:", e)

print(f"[{time.strftime('%H:%M:%S')}] CELL FULLY COMPLETE")

[20:34:02] starting learn()
Logging to ./tb_logs/peg_insert_side_ensemble_t2s\ens_t2s_run2_0
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 321      |
| time/              |          |
|    episodes        | 8        |
|    fps             | 233      |
|    time_elapsed    | 4        |
|    total_timesteps | 6000     |
| train/             |          |
|    actor_loss      | -15.7    |
|    critic_loss     | 0.402    |
|    ent_coef        | 0.745    |
|    ent_coef_loss   | -1.99    |
|    learning_rate   | 0.0003   |
|    n_updates       | 983      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 321      |
| time/              |          |
|    episodes        | 12       |
|    fps             | 232      |
|    time_elapsed    | 4        |
|    total_timesteps | 6000     |
-----------------------

d:\Miniconda3\envs\duke_rob\lib\site-packages\stable_baselines3\common\save_util.py:284: UserWarning: Path 'checkpoints\peg_insert_side_ensemble_t2s_2' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


**If it hangs again, read the last printed line:**

- `MODEL SAVED` but no `train env closed` → MuJoCo cleanup blocking.
  Harmless: the model is already on disk, interrupting loses nothing.
- No `learn() RETURNED` → the stall is inside training itself, not cleanup.
  Check whether TensorBoard scalars are still advancing to tell a genuine
  hang from very slow progress.
- Nothing after `starting learn()` for a long time → likely just the
  ensemble slowdown; compare against the smoke-test extrapolation above.

## 7. Evaluate the trained policy

In [22]:
eval_policy = SAC.load(os.path.join(CKPT_DIR, "sac_peg_insert_ensemble_t2s_final"))

N_EVAL = 20
success_steps = []
for seed in range(N_EVAL):
    env = make_env(seed=seed)
    obs, _ = env.reset()
    success_step = None
    for t in range(500):
        action, _ = eval_policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if info.get(SUCCESS_KEY, 0) and success_step is None:
            success_step = t
        if terminated or truncated:
            break
    env.close()
    success_steps.append(success_step)

success_rate = np.mean([s is not None for s in success_steps])
times = [s for s in success_steps if s is not None]
print(f"Success rate over {N_EVAL} seeds: {success_rate:.2%}")
if times:
    print(f"Mean time-to-success: {np.mean(times):.1f} steps")

Success rate over 20 seeds: 0.00%


In [23]:
def record_rollout(model, env, out_path, max_steps=500):
    obs, _ = env.reset()
    frames, success_step = [], None
    for t in range(max_steps):
        frames.append(env.render())
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if info.get(SUCCESS_KEY, 0) and success_step is None:
            success_step = t
        if terminated or truncated:
            break
    imageio.mimsave(out_path, frames, fps=20)
    return success_step, len(frames)

render_env = make_env(seed=0, render_mode="rgb_array")
ss, nf = record_rollout(eval_policy, render_env, "ensemble_t2s_behavior.mp4")
render_env.close()
print(f"Rollout saved: success_step={ss}, frames={nf}")

Rollout saved: success_step=None, frames=500


In [ ]:
from IPython.display import Video
Video("ensemble_t2s_behavior.mp4", embed=True)

## 8. Compare against stage 1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

with open("./checkpoints/peg_insert_side/eval_history.json") as f:
    orig_history = json.load(f)
with open(os.path.join(CKPT_DIR, "eval_history.json")) as f:
    ens_history = json.load(f)

orig_df = pd.DataFrame(orig_history)
ens_df = pd.DataFrame(ens_history)

plt.figure(figsize=(7,4))
plt.plot(orig_df["step"], orig_df["success_rate"], label="original (dense reward)")
plt.plot(ens_df["step"], ens_df["success_rate"], label="ensemble time2success only")
plt.xlabel("step"); plt.ylabel("success rate"); plt.legend()
plt.title("Original vs. ensemble time2success reward")
plt.show()